#### Genai app -

 we will source data from a website and then chunk it adn then embedd and make a FASSI db and then make a retreiver and invoke ollama llm through it 

In [2]:
import os 
from dotenv import load_dotenv

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"    
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = "Test"

In [3]:
#to read content from website using beautifulsoup4

from langchain_community.document_loaders import WebBaseLoader


e:\AAI\LangChain1\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
loader = WebBaseLoader("https://docs.python.org/3/library/datetime.html")
loader

In [13]:
docs = loader.load()
print(docs[0].page_content)


















datetime — Basic date and time types — Python 3.14.5rc1 documentation




















































    Theme
    
Auto
Light
Dark



Table of Contents

datetime — Basic date and time types
Aware and naive objects
Constants
Available types
Common properties
Determining if an object is aware or naive


timedelta objects
Examples of usage: timedelta


date objects
Examples of usage: date


datetime objects
Examples of usage: datetime


time objects
Examples of usage: time


tzinfo objects
timezone objects
strftime() and strptime() behavior
strftime() and strptime() format codes
Technical detail







Previous topic
Data Types


Next topic
zoneinfo — IANA time zone support



This page

Report a bug
Improve this page

Show source
        







Navigation


index

modules |

next |

previous |

Python »







3.14.5rc1 Documentation »
    
The Python Standard Library »
Data Types »
datetime — Basic date and time types







                     |

In [ ]:
# Load Data --> Process Data to docs --> Divide Docs to Chunks --> Create Vector Store --> Create RetrievalQA Chain --> Ask Questions

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.python.org/3/library/datetime.html', 'title': 'datetime — Basic date and time types — Python 3.14.5rc1 documentation', 'description': 'Source code: Lib/datetime.py The datetime module supplies classes for manipulating dates and times. While date and time arithmetic is supported, the focus of the implementation is on efficient attr...', 'language': 'en'}, page_content='datetime — Basic date and time types — Python 3.14.5rc1 documentation\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n    Theme\n    \nAuto\nLight\nDark\n\n\n\nTable of Contents\n\ndatetime — Basic date and time types\nAware and naive objects\nConstants\nAvailable types\nCommon properties\nDetermining if an object is aware or naive\n\n\ntimedelta objects\nExamples of usage: timedelta\n\n\ndate objects\nExamples of usage: date\n\n\ndatetime objects\nExamples of usage: datetime\n\n\ntime objects\nExamples of usage: time\n\n\

In [17]:
#Next step is to convert them to vectors and store them in a vector database. We will use FAISS as our vector database and Ollama as our embedding model.
# cosine similarity is a measure of similarity between two non-zero vectors in an inner product space. It is defined as the cosine of the angle between them, which ranges from -1 to 1. A cosine similarity of 1 indicates that the two vectors are identical, while a cosine similarity of -1 indicates that they are completely opposite. A cosine similarity of 0 indicates that the two vectors are orthogonal, meaning they have no similarity.
# cosine similarity in our application is used for comparing the similarity between the query vector and the document vectors in the vector database. When a user asks a question, we convert the question into a vector using the same embedding model that we used to convert the documents into vectors. Then, we calculate the cosine similarity between the query vector and each document vector in the vector database. The documents with the highest cosine similarity scores are considered the most relevant to the user's query and are retrieved for further processing by the language model.


In [20]:
from langchain_community.embeddings import OllamaEmbeddings

In [21]:
Embeddings = OllamaEmbeddings(model="gemma:2b")

C:\Users\TEJA\AppData\Local\Temp\ipykernel_11428\190043777.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  Embeddings = OllamaEmbeddings(model="gemma:2b")


In [22]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, Embeddings)
vectorstore

In [24]:
#similarity search for the query "What is datetime module in python?" and get the top 3 most similar documents
query = "What is datetime module in python?"
results = vectorstore.similarity_search_with_score(query)
results

[(Document(id='60ef2677-a358-466b-8806-3ba30a3d406f', metadata={'source': 'https://docs.python.org/3/library/datetime.html', 'title': 'datetime — Basic date and time types — Python 3.14.5rc1 documentation', 'description': 'Source code: Lib/datetime.py The datetime module supplies classes for manipulating dates and times. While date and time arithmetic is supported, the focus of the implementation is on efficient attr...', 'language': 'en'}, page_content='Instance methods:\n\n\ndatetime.date()¶\nReturn date object with same year, month and day.\n\n\n\ndatetime.time()¶\nReturn time object with same hour, minute, second, microsecond and fold.\ntzinfo is None. See also method timetz().\n\nChanged in version 3.6: The fold value is copied to the returned time object.\n\n\n\n\ndatetime.timetz()¶\nReturn time object with same hour, minute, second, microsecond, fold, and\ntzinfo attributes. See also method time().\n\nChanged in version 3.6: The fold value is copied to the returned time object.'

In [29]:
results[0][0].page_content

'Instance methods:\n\n\ndatetime.date()¶\nReturn date object with same year, month and day.\n\n\n\ndatetime.time()¶\nReturn time object with same hour, minute, second, microsecond and fold.\ntzinfo is None. See also method timetz().\n\nChanged in version 3.6: The fold value is copied to the returned time object.\n\n\n\n\ndatetime.timetz()¶\nReturn time object with same hour, minute, second, microsecond, fold, and\ntzinfo attributes. See also method time().\n\nChanged in version 3.6: The fold value is copied to the returned time object.'

In [47]:
#to create a document chain we need to import the LLM and create a chain that will take the retrieved documents and the query as input and return the answer as output

from langchain_community.llms import Ollama
llm = Ollama(model="gemma:2b")

C:\Users\TEJA\AppData\Local\Temp\ipykernel_11428\3748680224.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma:2b")


In [50]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Use from_template for a single string block
prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant that answers questions based on the following retrieved documents:
    <content>
    {context}
    </content>
    
    Question: {input}"""
)

document_chain = create_stuff_documents_chain(llm, prompt)

In [53]:
##Retrieval Chain
# Retreiver is a kind of interface that allows us to retrieve relevant documents from a vector database based on a query. It takes a query as input and returns a list of relevant documents along with their similarity scores. The retriever uses the same embedding model that we used to convert the documents into vectors to convert the query into a vector. Then, it calculates the cosine similarity between the query vector and each document vector in the vector database to determine which documents are most relevant to the query. The retriever is an essential component of a RetrievalQA chain, as it allows us to efficiently retrieve relevant information from a large collection of documents to answer user queries.

retreiver = vectorstore.as_retriever() # this will convert our vectorstore into a retriever that we can use in our RetrievalQA chain

from langchain_classic.chains import create_retrieval_chain
retreiver_chain = create_retrieval_chain(retreiver,document_chain) # this will create a retrieval chain that we can use to answer questions based on the documents in our vectorstore


In [55]:
##Get the answer for the question "What is datetime module in python?" using the retrieval chain we just created. The retrieval chain will first retrieve the relevant documents from the vectorstore based on the query and then pass those documents along with the query to the document chain to get the final answer.
query = "What is datetime module in python?"
answer = retreiver_chain.invoke({"input": query})
print(answer['answer'])

Sure, here's the answer to your question:

The datetime module provides classes for manipulating dates and times. It focuses on efficient attribute extraction for output formatting and manipulation.


In [56]:
answer

{'input': 'What is datetime module in python?',
 'context': [Document(id='60ef2677-a358-466b-8806-3ba30a3d406f', metadata={'source': 'https://docs.python.org/3/library/datetime.html', 'title': 'datetime — Basic date and time types — Python 3.14.5rc1 documentation', 'description': 'Source code: Lib/datetime.py The datetime module supplies classes for manipulating dates and times. While date and time arithmetic is supported, the focus of the implementation is on efficient attr...', 'language': 'en'}, page_content='Instance methods:\n\n\ndatetime.date()¶\nReturn date object with same year, month and day.\n\n\n\ndatetime.time()¶\nReturn time object with same hour, minute, second, microsecond and fold.\ntzinfo is None. See also method timetz().\n\nChanged in version 3.6: The fold value is copied to the returned time object.\n\n\n\n\ndatetime.timetz()¶\nReturn time object with same hour, minute, second, microsecond, fold, and\ntzinfo attributes. See also method time().\n\nChanged in version 